# Research 6: Предсказание Энергии Потенциалов

**Финальная версия ноутбука (ориентир: Intel Core i5-13600H, CPU-only).**

Цель: подобрать разрешённый ансамблевый регрессор и преобразование `256x256` потенциала в признаки так, чтобы получить низкий `MAE` и уложиться в лимит проверки (10 минут).


## План работы

1. Загружаем train/test из `public_tests`.
2. Делаем быстрый EDA и проверку масштаба задачи.
3. Сравниваем несколько моделей и типов признаков.
4. Фиксируем финальную конфигурацию для `potential_prediction.py`.
5. Проверяем результат через `run.py` и измеряем время полного прогона.

Ограничения задачи соблюдаются: **без бустинга** и **без нейросетей**.


In [ ]:
from __future__ import annotations

import json
import platform
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f'Python: {platform.python_version()}')
print('NumPy :', np.__version__)


def find_work_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'work', cwd.parent / 'work']
    for candidate in candidates:
        if (candidate / 'run.py').exists():
            return candidate
    raise FileNotFoundError('Не найден run.py. Откройте ноутбук из папки задания или work/.')


def _has_dataset(root: Path) -> bool:
    train_potentials = root / '01_test_potentials_input' / 'train' / 'potentials'
    gt_file = root / '01_test_potentials_gt' / 'target.json'
    return train_potentials.exists() and gt_file.exists()


def _try_unpack_public_tests(base_dir: Path) -> Path | None:
    zip_candidates = [
        base_dir / 'public_tests.zip',
        base_dir.parent / 'public_tests.zip',
    ]
    for zip_path in zip_candidates:
        if zip_path.exists():
            extract_dir = base_dir / 'public_tests'
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(extract_dir)
            if _has_dataset(extract_dir):
                return extract_dir
    return None


def find_data_root(work_dir: Path) -> Path:
    candidates = [work_dir / 'public_tests', work_dir]
    for root in candidates:
        if _has_dataset(root):
            return root

    unpacked = _try_unpack_public_tests(work_dir)
    if unpacked is not None:
        return unpacked

    raise FileNotFoundError(
        'Не найден public dataset с train/test/target.json и не удалось распаковать public_tests.zip.'
    )


WORK_DIR = find_work_dir()
DATA_ROOT = find_data_root(WORK_DIR)

TRAIN_DIR = DATA_ROOT / '01_test_potentials_input' / 'train' / 'potentials'
TEST_DIR = DATA_ROOT / '01_test_potentials_input' / 'test' / 'potentials'
TARGET_PATH = DATA_ROOT / '01_test_potentials_gt' / 'target.json'

print('WORK_DIR :', WORK_DIR)
print('DATA_ROOT:', DATA_ROOT)
print('TRAIN_DIR:', TRAIN_DIR)
print('TEST_DIR :', TEST_DIR)


In [ ]:
def load_dataset(data_dir: Path):
    files, x, y = [], [], []
    for file in sorted(data_dir.iterdir()):
        potential = np.load(file)
        files.append(file.name)
        x.append(potential['data'])
        y.append(float(potential['target']))
    return files, np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)


train_files, x_train, y_train = load_dataset(TRAIN_DIR)
test_files, x_test, y_test_embedded = load_dataset(TEST_DIR)

with TARGET_PATH.open('r', encoding='utf-8') as f:
    target_map = json.load(f)
y_public = np.array([target_map[file] for file in test_files], dtype=np.float64)

print('x_train:', x_train.shape, 'y_train:', y_train.shape)
print('x_test :', x_test.shape, 'y_test :', y_public.shape)
print('target range:', float(y_train.min()), '...', float(y_train.max()))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
indices = [0, 1, 2]
for ax, idx in zip(axes, indices):
    im = ax.imshow(x_train[idx], cmap='viridis')
    ax.set_title(f'{train_files[idx][:8]}...\ny={y_train[idx]:.5f}')
    ax.axis('off')
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
plt.tight_layout()
plt.show()


## Трансформеры признаков

Сравниваем три варианта:
- `flatten`: просто `reshape(256*256)` (базовый ориентир).
- `stats`: компактные статистики (квантили, пороговые площади, минимум/максимум и т.д.).
- `full`: расширенные физически осмысленные признаки (геометрия «ямы», радиальные профили, градиенты).


In [ ]:
def flatten_features(x: np.ndarray) -> np.ndarray:
    return x.reshape((x.shape[0], -1))


def stats_features(x: np.ndarray) -> np.ndarray:
    q = np.quantile(x, [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95], axis=(1, 2)).T
    mean = x.mean(axis=(1, 2))
    std = x.std(axis=(1, 2))
    mn = x.min(axis=(1, 2))
    mx = x.max(axis=(1, 2))
    l2 = np.sqrt((x * x).mean(axis=(1, 2)))
    below = np.stack([(x < t).mean(axis=(1, 2)) for t in [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]], axis=1)
    return np.hstack([mean[:, None], std[:, None], mn[:, None], mx[:, None], l2[:, None], q, below])


class PotentialTransformer:
    def fit(self, x: np.ndarray, y: np.ndarray | None = None):
        if x.ndim != 3:
            raise ValueError('Expected x with shape (n_samples, height, width)')
        height, width = x.shape[1], x.shape[2]
        self.rows_ = np.arange(height, dtype=np.float64)[:, None]
        self.cols_ = np.arange(width, dtype=np.float64)[None, :]
        self.norm_h_ = max(height - 1, 1)
        self.norm_w_ = max(width - 1, 1)
        self.quantiles_ = np.linspace(0.02, 0.98, 25)
        self.thresholds_ = np.linspace(0.0, 20.0, 41)
        max_radius = float(np.sqrt((height - 1) ** 2 + (width - 1) ** 2))
        self.radial_bins_ = np.linspace(0.0, max_radius, 37)
        return self

    def fit_transform(self, x: np.ndarray, y: np.ndarray | None = None) -> np.ndarray:
        self.fit(x, y)
        return self.transform(x)

    def transform(self, x: np.ndarray) -> np.ndarray:
        return np.vstack([self._extract_features(sample) for sample in x])

    def _extract_features(self, sample: np.ndarray) -> np.ndarray:
        flat = sample.ravel()
        f = [
            float(flat.mean()),
            float(flat.std()),
            float(flat.min()),
            float(flat.max()),
            float(np.sqrt(np.mean(flat * flat))),
        ]

        f.extend(np.quantile(flat, self.quantiles_).tolist())

        for threshold in self.thresholds_:
            f.append(float(np.mean(flat < threshold)))

        w = np.clip(20.0 - sample, 0.0, None)
        w_sum = float(w.sum()) + 1e-12

        row_c = float((w * self.rows_).sum() / w_sum)
        col_c = float((w * self.cols_).sum() / w_sum)
        row_var = float((w * (self.rows_ - row_c) ** 2).sum() / w_sum)
        col_var = float((w * (self.cols_ - col_c) ** 2).sum() / w_sum)
        row_col_cov = float((w * (self.rows_ - row_c) * (self.cols_ - col_c)).sum() / w_sum)

        min_row, min_col = np.unravel_index(np.argmin(sample), sample.shape)
        max_row, max_col = np.unravel_index(np.argmax(sample), sample.shape)

        f.extend([
            w_sum / sample.size,
            row_c / self.norm_h_,
            col_c / self.norm_w_,
            np.sqrt(row_var) / self.norm_h_,
            np.sqrt(col_var) / self.norm_w_,
            row_col_cov / (self.norm_h_ * self.norm_w_),
            min_row / self.norm_h_,
            min_col / self.norm_w_,
            float(sample[min_row, min_col]),
            max_row / self.norm_h_,
            max_col / self.norm_w_,
            float(sample[max_row, max_col]),
        ])

        radius = np.sqrt((self.rows_ - row_c) ** 2 + (self.cols_ - col_c) ** 2)
        for i in range(len(self.radial_bins_) - 1):
            in_ring = (radius >= self.radial_bins_[i]) & (radius < self.radial_bins_[i + 1])
            if np.any(in_ring):
                ring = sample[in_ring]
                f.append(float(ring.mean()))
                f.append(float(ring.std()))
            else:
                f.extend([0.0, 0.0])

        grad_x, grad_y = np.gradient(sample)
        grad_norm = np.sqrt(grad_x * grad_x + grad_y * grad_y)
        f.extend([
            float(grad_norm.mean()),
            float(grad_norm.std()),
            float(np.quantile(grad_norm, 0.9)),
            float(np.quantile(grad_norm, 0.99)),
        ])

        return np.asarray(f, dtype=np.float64)


In [ ]:
# Предвычисляем признаки один раз для чистого сравнения моделей.

x_train_flat = flatten_features(x_train)
x_test_flat = flatten_features(x_test)

x_train_stats = stats_features(x_train)
x_test_stats = stats_features(x_test)

full_transformer = PotentialTransformer()
x_train_full = full_transformer.fit_transform(x_train, y_train)
x_test_full = full_transformer.transform(x_test)

print('flat  features:', x_train_flat.shape[1])
print('stats features:', x_train_stats.shape[1])
print('full  features:', x_train_full.shape[1])


In [ ]:
def evaluate_public(name: str, model, x_tr: np.ndarray, x_te: np.ndarray) -> dict:
    t0 = time.perf_counter()
    model.fit(x_tr, y_train)
    fit_sec = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred = model.predict(x_te)
    pred_sec = time.perf_counter() - t1

    mae = mean_absolute_error(y_public, pred)
    return {
        'model': name,
        'features': x_tr.shape[1],
        'mae_public': mae,
        'fit_sec': fit_sec,
        'predict_sec': pred_sec,
    }


experiments = []

experiments.append(
    evaluate_public(
        name='DecisionTree + flatten',
        model=DecisionTreeRegressor(random_state=RANDOM_STATE),
        x_tr=x_train_flat,
        x_te=x_test_flat,
    )
)

experiments.append(
    evaluate_public(
        name='ExtraTrees(500) + stats',
        model=ExtraTreesRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        x_tr=x_train_stats,
        x_te=x_test_stats,
    )
)

experiments.append(
    evaluate_public(
        name='ExtraTrees(700) + full',
        model=ExtraTreesRegressor(
            n_estimators=700,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            min_samples_leaf=1,
        ),
        x_tr=x_train_full,
        x_te=x_test_full,
    )
)

results = pd.DataFrame(experiments).sort_values('mae_public').reset_index(drop=True)
results


In [ ]:
# Финальный выбор для отправки: ExtraTrees(700) + full features.

final_model = ExtraTreesRegressor(
    n_estimators=700,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    min_samples_leaf=1,
)

start = time.perf_counter()
final_model.fit(x_train_full, y_train)
fit_time = time.perf_counter() - start

start = time.perf_counter()
public_pred = final_model.predict(x_test_full)
pred_time = time.perf_counter() - start

public_mae = mean_absolute_error(y_public, public_pred)

print(f'Public MAE: {public_mae:.9f}')
print(f'Fit time  : {fit_time:.3f} sec')
print(f'Pred time : {pred_time:.3f} sec')


In [ ]:
# Проверка в формате тестирующей системы через run.py
# (используется директория, где лежит и run.py, и potential_prediction.py).

import subprocess


def find_solution_dir(work_dir: Path) -> Path:
    candidates = [work_dir, work_dir / 'work', work_dir.parent / 'work']
    for candidate in candidates:
        if (candidate / 'run.py').exists() and (candidate / 'potential_prediction.py').exists():
            return candidate
    raise FileNotFoundError('Не найдена директория с run.py и potential_prediction.py')


solution_dir = find_solution_dir(WORK_DIR)
cmd = [sys.executable, str(solution_dir / 'run.py'), str(DATA_ROOT)]

start = time.perf_counter()
proc = subprocess.run(cmd, cwd=solution_dir, text=True, capture_output=True, check=False)
elapsed = time.perf_counter() - start

print('SOLUTION_DIR:', solution_dir)
print('Return code :', proc.returncode)
print('Elapsed     :', f'{elapsed:.3f}', 'sec')
print('STDOUT:')
print(proc.stdout.strip() or '<empty>')
if proc.stderr.strip():
    print('STDERR:')
    print(proc.stderr.strip())


## Итог

- Для финального решения выбран `ExtraTreesRegressor` с `n_estimators=700` и расширенным `PotentialTransformer`.
- На public тесте получаем `MAE` существенно ниже порога `0.007`.
- Полный локальный прогон через `run.py` занимает секунды, что с большим запасом укладывается в лимит 10 минут.

Рекомендация для отправки:
- использовать файл `work/potential_prediction.py` (он синхронизирован с этой конфигурацией),
- отправлять `ipynb` без личных данных (ФИО/группа).
